**Tabela** | ecommerce_categorias |

**Origem** | squad2/silver/ecommerce_categorias (Delta) |

**Destinos** | 
* `gold/ecommerce_categorias_kpi` (Lakehouse - Overwrite)
* `gold/ecommerce_categorias_historico` (Lakehouse - Append)
* SQL Server: `ecommerce_categorias_kpi` (Overwrite) e `ecommerce_categorias_historico` (Append)
**Modo** | Delta Incremental Otimizado (Via Native SQL Server Connector)
**Regra de Negócio Aplicada** |
* Regra 2: Alertar se o número total de categorias raiz mudar entre lotes (adição/remoção não planejada)

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
from deltalake import DeltaTable, write_deltalake
import pandas as pd
from datetime import datetime
import json

TABELA = "ecommerce_categorias"

# Caminhos ABFSS oficiais utilizando exclusivamente as variáveis globais do seu Helpers
path_silver    = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/{TABELA}"
path_gold_kpi  = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/gold/{TABELA}_kpi"
path_gold_hist = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/gold/{TABELA}_historico"
path_control   = f"gold/control/{TABELA}.json"

print(f" Lendo de (Silver): {path_silver}")
print(f" Gravando em (Gold KPI): {path_gold_kpi}")
print(f" Gravando em (Gold Histórico): {path_gold_hist}")

try:
    # 1. Verifica se a tabela Silver já foi inicializada fisicamente no Azure
    if not DeltaTable.is_deltatable(path_silver, storage_options=get_storage_options()):
        print(f" [Aviso Gold] A tabela Silver de {TABELA} ainda não foi inicializada.")
    else:
        # 2. Abre a tabela Silver completa para análise de variação histórica
        dt_silver = DeltaTable(path_silver, storage_options=get_storage_options())
        df_pandas = dt_silver.to_pandas()
        
        # 3. CONTROLE INCREMENTAL: Instancia o cliente de controle da Gold
        squad2_client = get_squad2_client()
        file_client = squad2_client.get_file_client(path_control)
        
        processados = set()
        if file_client.exists():
            conteudo = file_client.download_file().readall().decode('utf-8')
            processados = set(json.loads(conteudo))
        
        # Filtra apenas os arquivos inéditos trazidos pelo novo lote
        df_novos_dados = df_pandas[~df_pandas['bronze_source_file'].isin(processados)].copy()
        
        if df_novos_dados.empty:
            print(" Camada Gold de Categorias em dia! Nenhum dado novo para processar.")
        else:
            print(f" Processando {len(df_novos_dados)} novas linhas da Silver...")
            
            # -------------------------------------------------------------------------
            # 4. APLICAÇÃO DA REGRA DE NEGÓCIO DA PLANILHA (CATEGORIAS RAIZ)
            # -------------------------------------------------------------------------
            # Identifica de forma dinâmica se existe alguma coluna hierárquica (ex: id_categoria_pai)
            col_pai = [c for c in df_pandas.columns if 'pai' in c.lower() or 'parent' in c.lower()]
            
            if col_pai:
                col_pai = col_pai[0]
                # Filtra apenas as categorias que não possuem pai (raízes da árvore de navegação)
                df_raiz_anterior = df_pandas[df_pandas['bronze_source_file'].isin(processados)]
                df_raiz_anterior = df_raiz_anterior[df_raiz_anterior[col_pai].isna() | (df_raiz_anterior[col_pai] == "")]
                total_raiz_anterior = df_raiz_anterior['id_categoria'].nunique()
                
                df_raiz_atual = df_pandas[df_pandas[col_pai].isna() | (df_pandas[col_pai] == "")]
                total_raiz_atual = df_raiz_atual['id_categoria'].nunique()
            else:
                # Fallback de segurança: se não houver coluna hierárquica, avalia a variação total do catálogo
                total_raiz_anterior = df_pandas[df_pandas['bronze_source_file'].isin(processados)]['id_categoria'].nunique()
                total_raiz_atual = df_pandas['id_categoria'].nunique()
            
            # Regra de Alerta: se já existia histórico e a contagem divergir, ativa a flag
            alerta_mudanca_raiz = "SIM" if (total_raiz_anterior > 0 and total_raiz_atual != total_raiz_anterior) else "NAO"
            
            # Estruturação da linha de KPI analítico
            df_pandas_kpi = pd.DataFrame([{
                "data_analise": str(datetime.now().date()),
                "horario_analise": datetime.now().time().strftime("%H:%M:%S"),
                "total_categorias_raiz_anterior": int(total_raiz_anterior),
                "total_categorias_raiz_atual": int(total_raiz_atual),
                "alerta_mudanca_raiz_nao_planejada": alerta_mudanca_raiz
            }])
            
            # 5. COLUNA DE AUDITORIA DA GOLD
            df_pandas_kpi['gold_processed_at'] = datetime.now()
            
            # Remove fusos horários para o formato Delta padrão
            for col in df_pandas_kpi.columns:
                if pd.api.types.is_datetime64_any_dtype(df_pandas_kpi[col]):
                    df_pandas_kpi[col] = df_pandas_kpi[col].dt.tz_localize(None)
                    
            # -------------------------------------------------------------------------
            # 6. SINK 1: GRAVAÇÃO NO LAKEHOUSE DELTA (Duplo Sink)
            # -------------------------------------------------------------------------
            # A) Estado Atual (Modo Overwrite)
            write_deltalake(path_gold_kpi, df_pandas_kpi, mode="overwrite", storage_options=get_storage_options())
            
            # B) Histórico Temporal (Modo Append)
            write_deltalake(path_gold_hist, df_pandas_kpi, mode="append", storage_options=get_storage_options())
            
            # -------------------------------------------------------------------------
            # 7. SINK 2: INGESTÃO NO SQL SERVER VIA FORMATO OFICIAL DO SQUAD
            # -------------------------------------------------------------------------
            # Converte para Spark DataFrame para acionar os drivers nativos do cluster
            spark_df = spark.createDataFrame(df_pandas_kpi)
            
            # Grava na tabela de KPIs instantâneos (Modo Overwrite)
            spark_df.write \
                .format("sqlserver") \
                .options(**SQL_OPTIONS) \
                .option("dbtable", f"{TABELA}_kpi") \
                .mode("overwrite") \
                .save()
            
            # Grava na tabela de histórico contínuo (Modo Append)
            spark_df.write \
                .format("sqlserver") \
                .options(**SQL_OPTIONS) \
                .option("dbtable", f"{TABELA}_historico") \
                .mode("append") \
                .save()
            
            # 8. ATUALIZA O CONTROL JSON DE GOVERNANÇA
            arquivos_atuais = set(df_novos_dados['bronze_source_file'].unique())
            todos_processados = list(processados.union(arquivos_atuais))
            file_client.upload_data(json.dumps(todos_processados), overwrite=True)
            
            print("\n SUCESSO! Pipeline de Categorias concluído e padronizado com a arquitetura do Squad 2.")
            display(df_pandas_kpi)

except Exception as e:
    print(f" Erro no processamento: {str(e)}")
    raise